# 第1周 b｜读懂医学数据表

学习目标：患者、一行一人、字段类型、单位、编码、原始文件与派生文件。

> **本周怎么学**：先学精讲 `w01a_first_look.ipynb`，再做这份**工程实践**。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：提交字段字典；解释Patient_ID为什么不能作预测变量。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 从结构开始，不急着训练模型
一行是一名合成人员；`cvd_event`和`cvd_time`必须联合解释。`Patient_ID`与`source_row_index`是读取时生成的连接标识，不是医学特征。以下可先查看表结构和少量记录；后续用于选择模型的探索只在训练集中进行。

In [2]:
display(df.head())
display(pd.DataFrame({"dtype":df.dtypes.astype(str), "missing":df.isna().sum()}))
print("患者编号唯一：", df.Patient_ID.is_unique)

,IRSD_quintile,Age,smoking_status,BMI,diabetes,CKD,HbA1c,eGFR,SBP,AF,cvd_event,cvd_time,source_row_index,Patient_ID
0,4,50.395265,non,28.166346,0,0,4.317890,83.077560,118.668194,0,1,2.963423,0,269
1,3,39.226761,ex,16.825992,0,0,4.700951,81.488401,123.719732,0,0,4.407252,1,272
2,5,55.489004,non,23.523419,0,0,3.669685,86.779230,126.650517,0,0,4.490739,2,281
3,4,51.910529,non,31.981932,0,0,4.486977,94.704093,113.889670,0,0,4.888965,3,296
4,1,47.091570,ex,25.351159,0,0,4.315440,86.336256,125.912615,0,0,4.490326,4,317


,dtype,missing
IRSD_quintile,int64,0
Age,float64,0
smoking_status,object,0
BMI,float64,0
diabetes,int64,0
CKD,int64,0
HbA1c,float64,0
eGFR,float64,0
SBP,float64,0
AF,int64,0


患者编号唯一： True


## 写自己的数据字典
不要把字段名当作全部信息。每一项至少记录：中文含义、单位、编码、获取时点、是否能作为基线预测变量。项目词典在 `docs/04_data_dictionary.md`。

In [3]:
dictionary = pd.DataFrame({
    "field": ["Age", "HbA1c", "cvd_event", "cvd_time", "Patient_ID"],
    "meaning": ["基线年龄", "糖化血红蛋白", "是否观察到CVD事件", "事件或删失前的观察时间", "连接用患者标识"],
    "unit_or_code": ["年", "%", "0/1", "年", "整数"],
    "baseline_predictor": [True, True, False, False, False]
})
display(dictionary)

,field,meaning,unit_or_code,baseline_predictor
0,Age,基线年龄,年,True
1,HbA1c,糖化血红蛋白,%,True
2,cvd_event,是否观察到CVD事件,0/1,False
3,cvd_time,事件或删失前的观察时间,年,False
4,Patient_ID,连接用患者标识,整数,False


In [4]:
from primecvd.data import audit_clean
display(pd.Series(audit_clean(df, cfg["horizon_years"])))

n                                                                                  50000
fields                                 [IRSD_quintile, Age, smoking_status, BMI, diab...
observed_events                                                                     2009
observed_event_fraction_not_5y_risk                                              0.04018
early_censored_before_horizon                                                      27542
followup_min                                                                    0.002427
followup_max                                                                    7.307214
at_risk_after_horizon                                                              20517
missing_by_field                       {'IRSD_quintile': 0, 'Age': 0, 'smoking_status...
dtype: object

## 独立练习
补齐其余字段的数据字典。任选一行，用自然语言解释这名合成人员的观察结果。不要将“未观察到事件”写成“终生不会得病”。参考答案只解释概念，见 `docs/07_exercises_and_answers.md`。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：不要把演示数据称为官方样本；不要覆盖raw文件。